<a href="https://colab.research.google.com/github/minseo-k3/ESAA_YB/blob/main/XGBoost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================================
# 1. 공통 전처리 파일(common.py) 정의 및 생성
# =====================================================================

import os
import urllib.request
import zipfile
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    StratifiedKFold,
    StratifiedGroupKFold,
    cross_val_score
)

# DACON 와인 품질 분류 데이터 URL
DATA_URL = 'https://bit.ly/3i4n1QB'

# 랜덤 시드 및 교차검증 설정
SEED = 42
N_SPLITS = 5


# 데이터 로드 함수
# train, test, sample_submission을 읽는다.
# 파일이 없으면 데이터를 다운로드한다.
def load_data(folder='data'):

    if not os.path.exists(os.path.join(folder, 'train.csv')):

        os.makedirs(folder, exist_ok=True)

        urllib.request.urlretrieve(DATA_URL, 'wine.zip')

        with zipfile.ZipFile('wine.zip') as z:
            z.extractall(folder)

    train = pd.read_csv(os.path.join(folder, 'train.csv'))
    test = pd.read_csv(os.path.join(folder, 'test.csv'))
    sub = pd.read_csv(os.path.join(folder, 'sample_submission.csv'))

    return train, test, sub


# 공통 전처리 함수
# 1. index 제거
# 2. type -> is_white (white=1, red=0)
# 3. 결합 이산화황 = 총 이산화황 - 유리 이산화황
# 4. 그 외에는 아무것도 하지 않는다.
def make_x(df):

    X = df.drop(
        columns=[c for c in ('index', 'quality') if c in df.columns]
    )

    X['is_white'] = (X['type'] == 'white').astype(int)

    X = X.drop(columns=['type'])

    X['bound_so2'] = (
        X['total sulfur dioxide'] - X['free sulfur dioxide']
    )

    return X


# 학습용 X와 y를 반환하는 함수
def make_xy(train):

    return make_x(train), train['quality']


# XGBoost용 라벨 변환 함수
# 기존 라벨 3~9를 0~6으로 변환한다.
# model.fit(X, shift_y(y))
# pred = unshift_y(model.predict(X_test))
def shift_y(y):

    return y - 3


# XGBoost 예측값을 원래 라벨 범위인 3~9로 복원한다.
def unshift_y(pred):

    return np.asarray(pred) + 3


# 공통 교차검증 설정
# 모든 모델에서 동일한 교차검증 방식을 사용한다.
def get_cv():

    return StratifiedKFold(
        N_SPLITS,
        shuffle=True,
        random_state=SEED
    )


# 최종 후보 모델 확인용 그룹 교차검증
# 성분이 같은 와인을 한 폴드에 묶는다.
# cv_g, groups = get_group_cv(train)
# cross_val_score(model, X, y, cv=cv_g, groups=groups)
def get_group_cv(train):

    feat = [
        c for c in train.columns
        if c not in ('index', 'quality')
    ]

    groups = train.groupby(feat, sort=False).ngroup()

    return (
        StratifiedGroupKFold(
            N_SPLITS,
            shuffle=True,
            random_state=SEED
        ),
        groups
    )


# 공통 교차검증 평가 함수
# shift=True이면 XGBoost용으로 라벨을 변환하여 평가한다.
def evaluate(
    model,
    X,
    y,
    name='',
    shift=False,
    verbose=True
):

    target = shift_y(y) if shift else y

    scores = cross_val_score(
        model,
        X,
        target,
        cv=get_cv(),
        n_jobs=1
    )

    if verbose:

        label = name or model.__class__.__name__

        print(
            '%-26s %.4f (±%.4f)'
            % (label, scores.mean(), scores.std())
        )

        print(
            '  폴드별 %s'
            % np.round(scores, 4).tolist()
        )

    return scores.mean(), scores


# 최종 후보 모델용 그룹 교차검증 평가 함수
# 처음 보는 성분 조합에 대한 성능을 확인한다.
# 일반 교차검증보다 점수가 낮게 나올 수 있다.
def evaluate_group(
    model,
    X,
    y,
    train,
    name='',
    shift=False
):

    cv_g, groups = get_group_cv(train)

    target = shift_y(y) if shift else y

    scores = cross_val_score(
        model,
        X,
        target,
        cv=cv_g,
        groups=groups,
        n_jobs=1
    )

    print(
        '%-26s %.4f (±%.4f)  [그룹 검증]'
        % (
            name or model.__class__.__name__,
            scores.mean(),
            scores.std()
        )
    )

    return scores.mean(), scores


# 모든 데이터를 6점으로 예측했을 때의 기준 점수
BASELINE = 0.4395


# 제출 파일 생성 및 형식 검사 함수
def make_submission(
    pred,
    sub,
    path='submission.csv'
):

    pred = np.asarray(pred)

    out = sub.copy()

    out['quality'] = pred

    problems = []

    # 제출 파일 열 이름 확인
    if list(out.columns) != ['index', 'quality']:

        problems.append(
            '열 이름이 [index, quality]가 아님: %s'
            % list(out.columns)
        )

    # 제출 파일 행 개수 확인
    if len(out) != 1000:

        problems.append(
            '행이 1000개가 아님: %d'
            % len(out)
        )

    # 예측값 정수형 여부 확인
    if not np.issubdtype(
        out['quality'].dtype,
        np.integer
    ):

        problems.append(
            'quality가 정수형이 아님: %s'
            % out['quality'].dtype
        )

    # 예측값 범위 확인
    lo, hi = out['quality'].min(), out['quality'].max()

    if lo < 3 or hi > 9:

        problems.append(
            'quality가 3~9 범위를 벗어남: %d ~ %d '
            '(XGBoost면 +3을 빠뜨렸는지 확인)'
            % (lo, hi)
        )

    # 검사 결과 출력
    if problems:

        print('[제출 파일 점검] 문제 발견')

        for p in problems:

            print('  - ' + p)

    else:

        out.to_csv(path, index=False)

        print(
            '[제출 파일 점검] 이상 없음 -> %s 저장'
            % path
        )

        print(
            out['quality']
            .value_counts()
            .sort_index()
            .to_string()
        )

    return out



In [ ]:
# =====================================================================
# 2. 데이터 로드 및 전처리 적용
# =====================================================================

# 데이터 로드
train, test, sub = load_data()

# 학습 데이터 전처리
X, y = make_xy(train)

# 테스트 데이터 전처리
X_test = make_x(test)

In [ ]:
# =====================================================================
# 3. XGBoost 모델 학습, 교차검증 및 제출 파일 생성
# =====================================================================

# XGBoost 분류 모델 라이브러리 불러오기
from xgboost import XGBClassifier


# XGBoost 모델 선언
xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    n_jobs=-1
)


# 교차검증 평가
# XGBoost는 라벨이 0부터 시작해야 하므로 shift=True 지정
evaluate(
    xgb_model,
    X,
    y,
    name='XGBoost',
    shift=True
)


# 전체 학습 데이터로 최종 모델 학습
# quality 라벨 3~9를 0~6으로 변환하여 학습
xgb_model.fit(
    X,
    shift_y(y)
)


# 테스트 데이터 예측
raw_preds = xgb_model.predict(X_test)


# 예측값을 원래 점수 범위인 3~9로 복원
final_preds = unshift_y(raw_preds)


# 제출 파일 생성 및 형식 검사
make_submission(
    final_preds,
    sub,
    path='xgb_submission.csv'
)

XGBoost                    0.6387 (±0.0110)
  폴드별 [0.65, 0.6409, 0.6451, 0.6397, 0.6178]
[제출 파일 점검] 이상 없음 -> xgb_submission.csv 저장
quality
4      8
5    357
6    499
7    124
8     12


,index,quality
0,0,5
1,1,6
2,2,5
3,3,5
4,4,6
...,...,...
995,995,5
996,996,6
997,997,5
998,998,6


In [ ]:
# =====================================================================
# 4. XGBoost 하이퍼파라미터 튜닝 및 성능 평가
# =====================================================================

import numpy as np
import pandas as pd
import time

from itertools import product
from xgboost import XGBClassifier

from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score


# =====================================================================
# 4-1. 확정된 교차검증 설정
# =====================================================================

SEED = 42
N_SPLITS = 5

# 기본 교차검증
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# 그룹 교차검증
cv_group = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# 기준선
BASELINE = 0.4395

# XGBoost 라벨 변환
y_shifted = shift_y(y)


# =====================================================================
# 4-2. 기존 XGBoost 모델 설정
# =====================================================================

baseline_params = {
    'n_estimators': 300,
    'learning_rate': 0.05,
    'max_depth': 6
}

baseline_model = XGBClassifier(
    **baseline_params,
    objective='multi:softprob',
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)


# =====================================================================
# 4-3. 하이퍼파라미터 탐색 범위
# =====================================================================

param_grid = {
    'n_estimators': [200, 300, 500],
    'learning_rate': [0.03, 0.05, 0.1],
    'max_depth': [4, 5, 6, 7],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# 전체 조합 개수 확인
total_combinations = np.prod([
    len(v) for v in param_grid.values()
])

print("전체 하이퍼파라미터 조합 수:", total_combinations)


# =====================================================================
# 4-4. 하이퍼파라미터 조합 생성
# =====================================================================

# 모든 조합 생성
param_names = list(param_grid.keys())

param_combinations = list(
    product(*[param_grid[name] for name in param_names])
)

# 기존 모델 설정을 첫 번째 후보로 추가
param_combinations.insert(
    0,
    tuple(baseline_params[name] for name in param_names[:3])
    + tuple([1, 0.8, 1.0])
)

# 중복 제거
param_combinations = list(dict.fromkeys(param_combinations))

print("실제 평가할 조합 수:", len(param_combinations))


# =====================================================================
# 4-5. 하이퍼파라미터 조합별 교차검증
# =====================================================================

results = []

start_time = time.time()

for i, params_tuple in enumerate(param_combinations, start=1):

    params = dict(zip(param_names, params_tuple))

    print("=" * 70)
    print(f"[{i}/{len(param_combinations)}] 평가 중")
    print("파라미터:", params)

    try:

        model = XGBClassifier(
            **params,
            objective='multi:softprob',
            eval_metric='mlogloss',
            random_state=42,
            n_jobs=-1
        )

        # 확정된 StratifiedKFold로 평가
        scores = cross_val_score(
            model,
            X,
            y_shifted,
            cv=cv,
            scoring='accuracy',
            n_jobs=1
        )

        mean_score = scores.mean()
        std_score = scores.std()

        # 결과 저장
        results.append({
            'params': params,
            'mean_accuracy': mean_score,
            'std_accuracy': std_score,
            'fold1': scores[0],
            'fold2': scores[1],
            'fold3': scores[2],
            'fold4': scores[3],
            'fold5': scores[4],
            'status': 'success',
            'error': None
        })

        print(f"평균 Accuracy: {mean_score:.4f}")
        print(f"표준편차: {std_score:.4f}")
        print("폴드별 점수:", np.round(scores, 4).tolist())

    except Exception as e:

        # 오류가 발생한 조합도 기록
        results.append({
            'params': params,
            'mean_accuracy': np.nan,
            'std_accuracy': np.nan,
            'fold1': np.nan,
            'fold2': np.nan,
            'fold3': np.nan,
            'fold4': np.nan,
            'fold5': np.nan,
            'status': 'failed',
            'error': str(e)
        })

        print("평가 실패:", str(e))


elapsed_time = time.time() - start_time

print("=" * 70)
print(f"총 평가 시간: {elapsed_time / 60:.2f}분")


# =====================================================================
# 4-6. 결과 정리 및 순위 확인
# =====================================================================

results_df = pd.DataFrame(results)

# 성공한 조합만 추출
success_df = results_df[
    results_df['status'] == 'success'
].copy()

# Accuracy가 높은 순서대로 정렬
success_df = success_df.sort_values(
    by='mean_accuracy',
    ascending=False
).reset_index(drop=True)

# 순위 추가
success_df['rank'] = np.arange(1, len(success_df) + 1)

# 최고 점수 확인
best_score = success_df.loc[0, 'mean_accuracy']

# 최적 하이퍼파라미터
best_params = success_df.loc[0, 'params']

print("=" * 70)
print("최적 하이퍼파라미터")
print(best_params)

print("=" * 70)
print(f"최고 평균 Accuracy: {best_score:.4f}")

print("최고 모델 폴드별 점수:")
print(
    success_df.loc[
        0,
        ['fold1', 'fold2', 'fold3', 'fold4', 'fold5']
    ].values.round(4)
)

print("=" * 70)
print(f"기준선 Accuracy: {BASELINE:.4f}")
print(f"기준선 대비 향상: {best_score - BASELINE:.4f}")


# =====================================================================
# 4-7. 탈락 조합 기록
# =====================================================================

# 최고 점수가 아닌 모든 조합을 탈락 후보로 기록
success_df['status'] = np.where(
    success_df['rank'] == 1,
    'best',
    'rejected'
)

success_df['rejection_reason'] = np.where(
    success_df['rank'] == 1,
    '',
    '최고 평균 Accuracy 조합보다 점수가 낮음'
)

# 실패한 조합도 별도 기록
failed_df = results_df[
    results_df['status'] == 'failed'
].copy()

failed_df['rank'] = np.nan
failed_df['rejection_reason'] = '모델 평가 중 오류 발생'

# 전체 기록 통합
all_results_df = pd.concat(
    [success_df, failed_df],
    ignore_index=True
)

# 결과 저장
all_results_df.to_csv(
    'xgb_tuning_results.csv',
    index=False,
    encoding='utf-8-sig'
)

print("=" * 70)
print("전체 튜닝 결과 저장: xgb_tuning_results.csv")
print("성공 조합 수:", len(success_df))
print("실패 조합 수:", len(failed_df))

전체 하이퍼파라미터 조합 수: 432
실제 평가할 조합 수: 432
[1/432] 평가 중
파라미터: {'n_estimators': 300, 'learning_rate': 0.05, 'max_depth': 6, 'min_child_weight': 1, 'subsample': 0.8, 'colsample_bytree': 1.0}
평균 Accuracy: 0.6534
표준편차: 0.0124
폴드별 점수: [0.6609, 0.6609, 0.6597, 0.657, 0.6288]
[2/432] 평가 중
파라미터: {'n_estimators': 200, 'learning_rate': 0.03, 'max_depth': 4, 'min_child_weight': 1, 'subsample': 0.8, 'colsample_bytree': 0.8}
평균 Accuracy: 0.5914
표준편차: 0.0080
폴드별 점수: [0.5936, 0.5836, 0.6051, 0.5914, 0.5833]
[3/432] 평가 중
파라미터: {'n_estimators': 200, 'learning_rate': 0.03, 'max_depth': 4, 'min_child_weight': 1, 'subsample': 0.8, 'colsample_bytree': 1.0}
평균 Accuracy: 0.5912
표준편차: 0.0086
폴드별 점수: [0.5918, 0.5836, 0.6069, 0.5905, 0.5833]
[4/432] 평가 중
파라미터: {'n_estimators': 200, 'learning_rate': 0.03, 'max_depth': 4, 'min_child_weight': 1, 'subsample': 1.0, 'colsample_bytree': 0.8}
평균 Accuracy: 0.5865
표준편차: 0.0105
폴드별 점수: [0.6064, 0.5773, 0.586, 0.5842, 0.5787]
[5/432] 평가 중
파라미터: {'n_estimators': 200, 'learning_r

최적 하이퍼파라미터
- 'n_estimators': 300
- 'learning_rate': 0.05
- 'max_depth': 7
- 'min_child_weight': 1
- 'subsample': 0.8
- 'colsample_bytree': 0.8
최고 평균 Accuracy: 0.6665

최고 모델 폴드별 점수:
[0.6764 0.6791 0.6624 0.677  0.6379]

기준선 Accuracy: 0.4395
기준선 대비 향상: 0.2270

전체 튜닝 결과 저장: xgb_tuning_results.csv
성공 조합 수: 432
실패 조합 수: 0

In [ ]:
# =====================================================================
# 5. 최종 후보 모델 그룹 교차검증
# =====================================================================

# 최적 파라미터로 모델 생성
best_model = XGBClassifier(
    **best_params,
    objective='multi:softprob',
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)


# =====================================================================
# 그룹 생성
# =====================================================================

feat = [
    c for c in train.columns
    if c not in ('index', 'quality')
]

gid = train.groupby(
    feat,
    sort=False
).ngroup()


# =====================================================================
# 그룹 교차검증
# =====================================================================

group_scores = cross_val_score(
    best_model,
    X,
    y_shifted,
    cv=cv_group,
    groups=gid,
    scoring='accuracy',
    n_jobs=1
)


# 그룹 검증 결과 출력
print("=" * 70)
print("최종 후보 모델 그룹 검증")

print(
    "Group CV 평균 Accuracy: %.4f"
    % group_scores.mean()
)

print(
    "Group CV 표준편차: %.4f"
    % group_scores.std()
)

print(
    "Group CV 폴드별 점수:",
    np.round(group_scores, 4).tolist()
)

print(
    "일반 CV 평균 Accuracy: %.4f"
    % best_score
)

print(
    "일반 CV와 그룹 CV 차이: %.4f"
    % (best_score - group_scores.mean())
)

print("=" * 70)

NameError: name 'y_shifted' is not defined

코드 오류는 저장 후 초기화되었을 때 다시 돌려본거라 무시해도 됨

In [ ]:
# =====================================================================
# 6. 최종 모델 학습 및 제출 파일 생성
# =====================================================================

# 최적 파라미터로 모델 생성
final_model = XGBClassifier(
    **best_params,
    objective='multi:softprob',
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)


# 전체 학습 데이터로 모델 학습
final_model.fit(
    X,
    y_shifted
)


# 테스트 데이터 예측
raw_preds = final_model.predict(X_test)


# 원래 품질 점수 3~9로 복원
final_preds = unshift_y(raw_preds).astype(int)


# 제출 파일 생성
make_submission(
    final_preds,
    sub,
    path='xgb_tuned_submission.csv'
)

NameError: name 'y_shifted' is not defined

제출 결과 0.682

In [ ]:
# Grid Search 스킵하고 검증된 고성능 파라미터 직접 지정
best_params = {
    'n_estimators': 1000,
    'learning_rate': 0.01,
    'max_depth': 6,
    'min_child_weight': 3,
    'subsample': 0.8,
    'colsample_bytree': 0.8
}

final_model = XGBClassifier(
    **best_params,
    objective='multi:softprob',
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)

final_model.fit(X, shift_y(y))
raw_preds = final_model.predict(X_test)
final_preds = unshift_y(raw_preds).astype(int)

make_submission(final_preds, sub, path='xgb_tuned_submission2.csv')

[제출 파일 점검] 이상 없음 -> xgb_tuned_submission2.csv 저장
quality
4      6
5    364
6    507
7    116
8      7


,index,quality
0,0,5
1,1,5
2,2,5
3,3,6
4,4,6
...,...,...
995,995,5
996,996,6
997,997,5
998,998,6


제출결과 0.662

In [ ]:
# Grid Search 스킵하고 과적합 방지 파라미터 적용
best_params = {
    'n_estimators': 300,        # 1000에서 300으로 줄여 과적합 방지
    'learning_rate': 0.03,      # 0.01보다 미세하게 올려 일반화 성능 확보
    'max_depth': 5,             # 복잡도를 낮추기 위해 6 -> 5로 변경 권장
    'min_child_weight': 3,
    'subsample': 0.8,
    'colsample_bytree': 0.8
}

final_model = XGBClassifier(
    **best_params,
    objective='multi:softprob',
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)

final_model.fit(X, shift_y(y))
raw_preds = final_model.predict(X_test)
final_preds = unshift_y(raw_preds).astype(int)

make_submission(final_preds, sub, path='xgb_tuned_submission3.csv')

[제출 파일 점검] 이상 없음 -> xgb_tuned_submission3.csv 저장
quality
4      4
5    370
6    513
7    106
8      7


,index,quality
0,0,5
1,1,5
2,2,5
3,3,6
4,4,6
...,...,...
995,995,5
996,996,6
997,997,5
998,998,6


제출결과 0.642

In [ ]:
# 1. 화면에서 찾은 최적 하이퍼파라미터 설정
best_params = {
    'n_estimators': 300,
    'learning_rate': 0.05,
    'max_depth': 7,
    'min_child_weight': 1,
    'subsample': 0.8,
    'colsample_bytree': 0.8
}

# 2. 최적 파라미터 기반 XGBoost 분류 모델 생성
final_model = XGBClassifier(
    **best_params,
    objective='multi:softprob',
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)

# 3. 전체 학습 데이터로 모델 학습
final_model.fit(X, shift_y(y))

# 4. 테스트 데이터 예측 및 복원 (+3 shift 복원)
raw_preds = final_model.predict(X_test)
final_preds = unshift_y(raw_preds).astype(int)

# 5. 제출 파일 생성 (덮어쓰기 방지를 위한 새 파일명 사용)
make_submission(final_preds, sub, path='xgb_best_tuned_submission.csv')

[제출 파일 점검] 이상 없음 -> xgb_best_tuned_submission.csv 저장
quality
4      8
5    353
6    505
7    120
8     14


,index,quality
0,0,5
1,1,5
2,2,5
3,3,5
4,4,6
...,...,...
995,995,5
996,996,6
997,997,5
998,998,6


제출결과 0.682 (맨처음 모델과 같은 것, 파일 이름만 다르게 저장)

In [ ]:
import numpy as np
from xgboost import XGBClassifier

# [조합 1 적용 코드]
best_params_new = {
    'n_estimators': 500,
    'learning_rate': 0.03,
    'max_depth': 8,
    'min_child_weight': 1,
    'subsample': 0.8,
    'colsample_bytree': 0.8
}

# 1. 조합 1 단일 모델 학습 및 제출
model_v1 = XGBClassifier(
    **best_params_new,
    objective='multi:softprob',
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)

model_v1.fit(X, shift_y(y))
preds_v1 = unshift_y(model_v1.predict(X_test)).astype(int)
make_submission(preds_v1, sub, path='xgb_depth8_lr03_submission.csv')


# [조합 4: 최고 조합(11번셀) 기반 5개 시드 앙상블 코드]
seeds = [42, 100, 200, 300, 777]
prob_list = []

base_params = {
    'n_estimators': 300,
    'learning_rate': 0.05,
    'max_depth': 7,
    'min_child_weight': 1,
    'subsample': 0.8,
    'colsample_bytree': 0.8
}

for s in seeds:
    m = XGBClassifier(**base_params, objective='multi:softprob', eval_metric='mlogloss', random_state=s, n_jobs=-1)
    m.fit(X, shift_y(y))
    prob_list.append(m.predict_proba(X_test))

# 5개 모델의 예측 확률 평균 후 최종 예측
avg_prob = np.mean(prob_list, axis=0)
ensemble_preds = unshift_y(np.argmax(avg_prob, axis=1)).astype(int)
make_submission(ensemble_preds, sub, path='xgb_seed_ensemble_submission.csv')

[제출 파일 점검] 이상 없음 -> xgb_depth8_lr03_submission.csv 저장
quality
4      7
5    353
6    508
7    118
8     14
[제출 파일 점검] 이상 없음 -> xgb_seed_ensemble_submission.csv 저장
quality
4      8
5    351
6    505
7    123
8     13


,index,quality
0,0,5
1,1,6
2,2,6
3,3,5
4,4,6
...,...,...
995,995,5
996,996,6
997,997,5
998,998,6


제출결과 0.695(xgb_depth8_lr03_submission.csv), 0.693(xgb_seed_ensemble_submission.csv)